# Continued retries and PTRM on Sudoku-Extreme

Two experiments on the 1,000 evaluation puzzles, with three seeds:

- up to 16 retries that continue the failed trajectory, without perturbation (TRM under the harness), with `y`
  reset or resampled, with `z` resampled or perturbed by PTRM's noise, or both, and a restart without perturbation;
- PTRM rerun on our checkpoint: 100 rollouts of D = 16 and D = 64 steps at σ = 0.3, read with PTRM's own protocol
  and under the harness, and the deterministic unroll of D = 64 steps.

Set `CKPT_TRM` and `CACHE_DIR` below. Passes and rollouts are saved in `CACHE_DIR` and reloaded when the notebook
runs again. `SudokuExtreme_GAR` and `SudokuExtreme_z_instead_of_y` read the PTRM rollouts from there.

In [ ]:
!pip -q install einops pydantic argdantic tqdm huggingface_hub
!test -d TRM || git clone -q https://github.com/SamsungSAILMontreal/TinyRecursiveModels TRM
!git -C TRM checkout -q c011037

In [ ]:
import os, sys, json, subprocess, dataclasses

import numpy as np
import pandas as pd
import torch
import yaml

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass

CKPT_TRM = "/content/drive/MyDrive/GAR/sudoku/checkpoints/step_65100"  # with all_config.yaml next to it
CACHE_DIR = "/content/drive/MyDrive/GAR/sudoku/cache"
DATA_PATH = "data/sudoku-extreme-1k-aug-1000"
GLOBAL_BATCH_SIZE = 768
DEVICE = "cuda"

sys.path.insert(0, os.path.abspath("TRM"))
from models.losses import ACTLossHead, IGNORE_LABEL_ID
from models.recursive_reasoning.trm import TinyRecursiveReasoningModel_ACTV1
from puzzle_dataset import PuzzleDataset, PuzzleDatasetConfig

os.makedirs(CACHE_DIR, exist_ok=True)
pd.options.display.float_format = "{:.2f}".format

## Checkpoint and test split

In [ ]:
if not os.path.exists(os.path.join(DATA_PATH, "test", "dataset.json")):
    subprocess.run([sys.executable, "TRM/dataset/build_sudoku_dataset.py", "--output-dir", DATA_PATH,
                    "--subsample-size", "1000", "--num-aug", "1000"], check=True)
test_set = PuzzleDataset(PuzzleDatasetConfig(seed=0, dataset_paths=[DATA_PATH], global_batch_size=GLOBAL_BATCH_SIZE,
                                             test_set_mode=True, epochs_per_iter=1, rank=0, num_replicas=1),
                         split="test")
test_set._lazy_load_dataset()
TEST = test_set._data["all"]

with open(os.path.join(os.path.dirname(CKPT_TRM), "all_config.yaml")) as f:
    arch = yaml.safe_load(f)["arch"]
loss_type = arch.pop("loss")["loss_type"]
del arch["name"]
state = {k.replace("_orig_mod.", "", 1): v for k, v in torch.load(CKPT_TRM, map_location=DEVICE).items()}
config = dict(arch, batch_size=GLOBAL_BATCH_SIZE, vocab_size=test_set.metadata.vocab_size,
              seq_len=test_set.metadata.seq_len,
              num_puzzle_identifiers=state["model.inner.puzzle_emb.weights"].shape[0])
with torch.device(DEVICE):
    model = ACTLossHead(TinyRecursiveReasoningModel_ACTV1(config), loss_type=loss_type)
model.load_state_dict(state, strict=False, assign=True)
model.eval()  # halting is off: every block runs exactly T supervision steps
inner = model.model.inner
T = model.model.config.halt_max_steps  # supervision steps per block (16)
P = inner.puzzle_emb_len  # prefix positions before the 81 cells (16)

## The harness

The standard pass runs one block of `T` supervision steps. The gate re-opens a puzzle whose halting logit `q` is
negative, and a retry runs one more block on it. The search stops at the first block with `q >= 0`, or after
`ROUNDS` retries, and returns the grid of highest `q` among the blocks run (the fallback).

A continued retry starts from the state of the failed block. At the block boundary, it can resample the fillable
cells of `y` (`z_H` in TRM's code) or reset them to `y_init`, resample the fillable cells or all cells of `z`
(`z_L`), and it can add PTRM's noise to `z` before every step of the block. Resampled cells receive isotropic
directions at the norm of `y_init` or `z_init`. `cold` restarts from the initial states without perturbation, which
replays the standard pass.

In [ ]:
Q_THR = 0.0  # the gate re-opens a puzzle when q < Q_THR, and a block with q >= Q_THR stops the search
BLANK = 1  # input token of the fillable cells
SIGMA = 0.3  # PTRM's noise scale on Sudoku-Extreme
ROUNDS = 16
SEEDS = [0, 1, 2]

# What a continued retry does to y and to z at the block boundary, and the noise it adds to z before every step.
ARMS = {"none": ("keep", "keep", 0.0),
        "randvec": ("resample", "keep", 0.0),
        "zvec": ("keep", "keep", SIGMA),
        "randvec_zvec": ("resample", "keep", SIGMA),
        "randz": ("keep", "resample", 0.0),
        "randz_all": ("keep", "resample all cells", 0.0),
        "randvec_randz": ("resample", "resample", 0.0),
        "retract": ("reset", "keep", 0.0),
        "cold": ("restart", "restart", 0.0)}


def clone(x):
    """Deep copy of a TRM carry."""
    if torch.is_tensor(x):
        return x.clone()
    if dataclasses.is_dataclass(x):
        return type(x)(**{f.name: clone(getattr(x, f.name)) for f in dataclasses.fields(x)})
    if isinstance(x, dict):
        return {k: clone(v) for k, v in x.items()}
    return x


def correct(pred, batch):
    """Exact match on the labelled cells, as counted by TRM's evaluator, and the rows that have labels."""
    mask = batch["labels"] != IGNORE_LABEL_ID
    return (mask & (pred == batch["labels"])).sum(-1) == mask.sum(-1), mask.any(-1)


@torch.no_grad()
def run_block(batch, carry=None, sigma=0.0, active=None, gen=None):
    """One block of T supervision steps; carry=None starts from the initial states (the standard pass). With
    sigma > 0, noise N(0, sigma^2 I) is added to z on the rows `active` before every step."""
    if carry is None:
        with torch.device(DEVICE):
            carry = model.initial_carry(batch)
    for _ in range(T):
        if sigma > 0:
            zL = carry.inner_carry.z_L
            e = torch.randn(zL.shape, generator=gen, device=zL.device, dtype=torch.float32)
            zL.copy_(torch.where(active.view(-1, 1, 1), (zL.float() + sigma * e).to(zL.dtype), zL))
        carry, _, _, out, _ = model(carry=carry, batch=batch, return_keys={"preds", "q_halt_logits"})
    return carry, out["preds"], out["q_halt_logits"].float()


def directions(shape, gen, init):
    """Isotropic random directions, one per position, at the norm of `init`."""
    e = torch.randn(shape, generator=gen, device=DEVICE, dtype=torch.float32)
    return e / e.norm(dim=-1, keepdim=True).clamp_min(1e-9) * init.float().norm()


@torch.no_grad()
def retry(carry, fill, active, arm, gen_y, gen_z):
    """Prepares a continued retry of the rows `active` at the block boundary."""
    y_op, z_op, _ = ARMS[arm]
    zH, zL = carry.inner_carry.z_H, carry.inner_carry.z_L
    cells = (fill & active.view(-1, 1)).unsqueeze(-1)
    if y_op == "reset":
        zH[:, P:] = torch.where(cells, inner.H_init.to(zH.dtype), zH[:, P:])
    elif y_op == "resample":
        zH[:, P:] = torch.where(cells, directions(zH[:, P:].shape, gen_y, inner.H_init).to(zH.dtype), zH[:, P:])
    if z_op != "keep":
        where = active.view(-1, 1, 1) if z_op == "resample all cells" else cells
        zL[:, P:] = torch.where(where, directions(zL[:, P:].shape, gen_z, inner.L_init).to(zL.dtype), zL[:, P:])
    carry.halted.fill_(False)  # so that the next step continues from this state instead of resetting it
    carry.steps.zero_()


@torch.no_grad()
def run_pass(seed):
    """The harness on the pool with each arm. Per puzzle, in pool order: the standard pass (ok0, q0, pred0) and, per
    arm, ok[:, r] (the returned grid is correct after r retries) and act[:, r] (the puzzle is open before retry
    r + 1)."""
    res = {"ok0": [], "q0": [], "pred0": [], **{arm + suffix: [] for arm in ARMS for suffix in ("__ok", "__act")}}
    for b, start in enumerate(range(0, POOL_N, GLOBAL_BATCH_SIZE)):
        batch = pool_batch(POOL_IDX[start:start + GLOBAL_BATCH_SIZE])
        fill = batch["inputs"] == BLANK
        carry0, pred0, q0 = run_block(batch)
        ok0, valid = correct(pred0, batch)
        ok0 &= valid
        rows = valid.nonzero(as_tuple=True)[0]  # drops the padding rows of the last batch
        for k, v in (("ok0", ok0), ("q0", q0), ("pred0", pred0)):
            res[k].append(v[rows].cpu().numpy())
        for arm, (_, _, sigma) in ARMS.items():
            gen_y = torch.Generator(device=DEVICE).manual_seed(seed * 100003 + b)
            gen_noise = torch.Generator(device=DEVICE).manual_seed(seed * 100003 + b + 7919)
            gen_z = torch.Generator(device=DEVICE).manual_seed(seed * 100003 + b + 15485863)
            carry = clone(carry0)
            best_p, best_q = pred0.clone(), q0.clone()
            active = valid & (q0 < Q_THR)
            ok, act = [ok0], []
            for r in range(ROUNDS):
                if arm == "cold" and r > 0:  # a restart without perturbation is run once
                    active = torch.zeros_like(active)
                act.append(active.clone())
                if active.any():
                    if arm == "cold":
                        carry, p, q = run_block(batch)
                    else:
                        retry(carry, fill, active, arm, gen_y, gen_z)
                        carry, p, q = run_block(batch, carry, sigma, active, gen_noise)
                    better = active & (q > best_q)
                    best_p = torch.where(better.view(-1, 1), p, best_p)
                    best_q = torch.where(better, q, best_q)
                    active &= q < Q_THR
                ok.append(correct(best_p, batch)[0] & valid)
            res[arm + "__ok"].append(torch.stack(ok, 1)[rows].cpu().numpy())
            res[arm + "__act"].append(torch.stack(act, 1)[rows].cpu().numpy())
    return {k: np.concatenate(v) for k, v in res.items()}

## The 1,000 evaluation puzzles and the cache

In [ ]:
POOL_N = 1000
POOL_IDX = np.sort(np.random.default_rng(0).choice(np.arange(len(TEST["inputs"])), size=POOL_N, replace=False))


def pool_batch(idx, copies=1):
    """The test puzzles `idx`, repeated `copies` times, as one padded batch of the test loader."""
    idx = np.tile(np.asarray(idx, dtype=np.int64), copies)
    pid = np.searchsorted(TEST["puzzle_indices"], idx, side="right") - 1
    rows = {"inputs": np.asarray(TEST["inputs"][idx]), "labels": np.asarray(TEST["labels"][idx]),
            "puzzle_identifiers": np.asarray(TEST["puzzle_identifiers"][pid])}
    return {k: v.to(DEVICE) for k, v in test_set._collate_batch(rows).items()}


# Every cached file must reproduce, bit for bit, the halting logits of this standard pass on the first batch.
ref_batch = pool_batch(POOL_IDX[:GLOBAL_BATCH_SIZE])
ref_carry, ref_pred, ref_q = run_block(ref_batch)
REF_Q0 = ref_q[:min(POOL_N, GLOBAL_BATCH_SIZE)].cpu().numpy()



def cached_pass(seed):
    """The pass of all arms for `seed`, loaded from the cache or run and saved."""
    path = os.path.join(CACHE_DIR, "TRM_pool_policy_seed%d.npz" % seed)
    if not os.path.exists(path):
        res = run_pass(seed)
        np.savez_compressed(path, pool_idx=POOL_IDX, q0=res.pop("q0"), pred0=res.pop("pred0").astype(np.int16),
                            **{k: v.astype(np.uint8) for k, v in res.items()})
    f = np.load(path)
    assert np.array_equal(f["pool_idx"], POOL_IDX) and np.array_equal(f["q0"][:len(REF_Q0)], REF_Q0), \
        "%s comes from other puzzles or another numerical setup (GPU, PyTorch): delete it to recompute it" % path
    return dict(ok0=f["ok0"].astype(bool), q0=f["q0"], pred0=f["pred0"],
                **{arm: dict(ok=f[arm + "__ok"].astype(bool), act=f[arm + "__act"].astype(bool)) for arm in ARMS})

## Continued retries

In [ ]:
PASSES = [cached_pass(s) for s in SEEDS]
ok0, q0 = PASSES[0]["ok0"], PASSES[0]["q0"]
assert all(np.array_equal(p["q0"], q0) and np.array_equal(p["pred0"], PASSES[0]["pred0"]) for p in PASSES)
N_FAIL = (~ok0).sum()
print("Standard pass: %.2f%% of %d puzzles, %d failures." % (100 * ok0.mean(), POOL_N, N_FAIL))
print("The gate re-opens %d of the %d failures (recall %.1f%%) and %d of the %d successes (false-positive rate %.1f%%)."
      % ((~ok0 & (q0 < Q_THR)).sum(), N_FAIL, 100 * (~ok0 & (q0 < Q_THR)).sum() / N_FAIL,
         (ok0 & (q0 < Q_THR)).sum(), ok0.sum(), 100 * (ok0 & (q0 < Q_THR)).sum() / ok0.sum()))
print("Norm of a position of y after the standard pass: %.3f (sqrt(d) = %.3f). Norm of y_init: %.3f, of z_init: %.3f."
      % (ref_carry.inner_carry.z_H[:, P:].float().norm(dim=-1).mean(), inner.H_init.numel() ** 0.5,
         inner.H_init.float().norm(), inner.L_init.float().norm()))


def summary(arm):
    """After ROUNDS retries of `arm`: failures recovered, accuracy, cost, successes lost."""
    ok = [p[arm]["ok"][:, ROUNDS] for p in PASSES]
    acc = [100 * v.mean() for v in ok]
    rec = [(~ok0 & v).sum() for v in ok]
    return {"recovered": np.mean(rec), "sd": np.std(rec, ddof=1), "per seed": tuple(int(x) for x in rec),
            "recovered (%)": 100 * np.mean(rec) / N_FAIL, "sd (%)": 100 * np.std(rec, ddof=1) / N_FAIL,
            "accuracy": np.mean(acc), "accuracy sd": np.std(acc, ddof=1),
            "cost": np.mean([1.0 + p[arm]["act"][:, :ROUNDS].sum() / POOL_N for p in PASSES]),
            "lost": np.mean([(ok0 & ~v).sum() for v in ok])}

After `ROUNDS` retries, per arm: the standard-pass failures recovered (mean, standard deviation and value per seed,
in counts and in % of the failures), the accuracy (%), the cost (blocks per puzzle) and the standard-pass successes
lost.

In [ ]:
LATTICE = pd.DataFrame.from_dict({arm: summary(arm) for arm in ARMS}, orient="index")
LATTICE

## PTRM

A rollout starts from the initial states and runs `D` supervision steps, each preceded by noise N(0, σ² I) on `z`.
Rollouts are packed in batches, `m` copies of `g` puzzles. Three checks come first: without noise, a rollout of `T`
steps reproduces the standard pass, alone or packed, and a rollout of `4T` steps reproduces four continued blocks.

In [ ]:
PTRM_K = 100
PTRM_D = [16, 64]


@torch.no_grad()
def rollout(batch, D, sigma, gen):
    """One PTRM rollout per row of `batch`; returns the grids and halting logits after D steps."""
    with torch.device(DEVICE):
        carry = model.initial_carry(batch)
    B = batch["inputs"].shape[0]
    carry.inner_carry = inner.reset_carry(torch.ones(B, dtype=torch.bool, device=DEVICE), carry.inner_carry)
    carry.halted = torch.zeros(B, dtype=torch.bool, device=DEVICE)
    carry.steps = torch.zeros(B, dtype=torch.int32, device=DEVICE)
    carry.current_data = {k: v.clone() for k, v in batch.items()}
    for _ in range(D):
        if sigma > 0:
            zL = carry.inner_carry.z_L
            e = torch.randn(zL.shape, generator=gen, device=zL.device, dtype=torch.float32)
            zL.copy_((zL.float() + sigma * e).to(zL.dtype))
        carry, _, _, out, _ = model(carry=carry, batch=batch, return_keys={"preds", "q_halt_logits"})
        carry.halted.fill_(False)
        carry.steps.zero_()
    return out["preds"], out["q_halt_logits"].float()


n = len(REF_Q0)
m = min(PTRM_K, GLOBAL_BATCH_SIZE)  # rollouts packed in one batch: m copies of g puzzles
g = max(1, GLOBAL_BATCH_SIZE // m)
gen = torch.Generator(device=DEVICE)
pred, q = rollout(ref_batch, T, 0.0, gen)
assert (pred[:n] == ref_pred[:n]).all() and (q[:n] == ref_q[:n]).all()
pred, q = rollout(pool_batch(POOL_IDX[:g], m), T, 0.0, gen)
assert all((pred[j * g:(j + 1) * g] == ref_pred[:g]).all() and (q[j * g:(j + 1) * g] == ref_q[:g]).all()
           for j in range(m))
carry = clone(ref_carry)  # the standard pass followed by three continued blocks
for _ in range(3):
    carry.halted.fill_(False)
    carry.steps.zero_()
    carry, pred4, q4 = run_block(ref_batch, carry)
pred, q = rollout(ref_batch, 4 * T, 0.0, gen)
assert (pred[:n] == pred4[:n]).all() and (q[:n] == q4[:n]).all()


@torch.no_grad()
def ptrm(K, D, sigma, seed):
    """K rollouts of D steps per pool puzzle, loaded from the cache or run and saved: correctness and halting
    logit of each rollout, arrays [K, POOL_N]."""
    path = os.path.join(CACHE_DIR, "TRM__K%d_D%d_s%.2f__seed%d.npz" % (K, D, sigma, seed))
    if not os.path.exists(path):
        m = min(K, GLOBAL_BATCH_SIZE)
        g = max(1, GLOBAL_BATCH_SIZE // m)
        gen = torch.Generator(device=DEVICE).manual_seed(seed * 1000003 + D * 31 + round(sigma * 1000))
        ok, qs = np.zeros((K, POOL_N), bool), np.zeros((K, POOL_N), np.float32)
        for start in range(0, POOL_N, g):
            idx = POOL_IDX[start:start + g]
            n = len(idx)
            batch = pool_batch(idx, m)
            labels = batch["labels"][:n]
            mask = labels != IGNORE_LABEL_ID
            for c in range(-(-K // m)):
                pred, q = rollout(batch, D, sigma, gen)
                for j in range(min(m, K - c * m)):
                    copy = slice(j * n, (j + 1) * n)
                    solved = (mask & (pred[copy] == labels)).sum(-1) == mask.sum(-1)
                    ok[c * m + j, start:start + n] = solved.cpu().numpy()
                    qs[c * m + j, start:start + n] = q[copy].cpu().numpy()
        np.savez_compressed(path, ok=ok, q=qs, pool_idx=POOL_IDX, ref_q0=REF_Q0)
    f = np.load(path)
    assert np.array_equal(f["pool_idx"], POOL_IDX) and np.array_equal(f["ref_q0"], REF_Q0), \
        "%s comes from other puzzles or another numerical setup (GPU, PyTorch): delete it to recompute it" % path
    return f["ok"].astype(bool), f["q"]


ROLLOUTS = {D: [ptrm(PTRM_K, D, SIGMA, s) for s in SEEDS] for D in PTRM_D}
UNROLL = ptrm(1, 64, 0.0, 0)[0][0]  # TRM without noise for 64 steps

### With PTRM's own protocol

The first `K'` rollouts of each configuration, with PTRM's selector (the rollout of highest `q`, *accuracy*) and the
oracle that accepts any correct rollout (*pass@K'*), then with the fallback, which also counts the standard pass as
a candidate and keeps it on ties. Cost in blocks per puzzle, means and standard deviations over seeds; *lost*
counts the standard-pass successes that a configuration fails.

In [ ]:
K_LIST = [k for k in (1, 5, 10, 25, 100) if k <= PTRM_K]


def read(ok, q, k):
    """Correctness per puzzle of the first k rollouts: PTRM's selector, pass@k, and the selector with the fallback."""
    ok, q = ok[:k], q[:k]
    cols = np.arange(ok.shape[1])
    fallback = np.concatenate([ok0[None, :], ok])[np.concatenate([q0[None, :], q]).argmax(0), cols]
    return ok[q.argmax(0), cols], ok.any(0), fallback


def accuracy(ok):
    """Mean and standard deviation over seeds of the accuracy (%), and mean of the standard-pass successes lost."""
    acc = [100 * v.mean() for v in ok]
    return np.mean(acc), np.std(acc, ddof=1), np.mean([(ok0 & ~v).sum() for v in ok])


rows = {}
for D in PTRM_D:
    for k in K_LIST:
        sel, pas, fb = zip(*(read(ok, q, k) for ok, q in ROLLOUTS[D]))
        (a, sd, lost), (pa, psd, _), (fa, fsd, flost) = accuracy(sel), accuracy(pas), accuracy(fb)
        rows[(k, D)] = {"cost": k * D / T, "accuracy": a, "sd": sd, "pass@K'": pa, "pass sd": psd, "lost": lost,
                        "fallback accuracy": fa, "fallback sd": fsd, "fallback lost": flost}
rows[("unroll", 64)] = {"cost": 64 / T, "accuracy": 100 * UNROLL.mean(), "lost": (ok0 & ~UNROLL).sum()}
PTRM_OWN = pd.DataFrame.from_dict(rows, orient="index").rename_axis(["K'", "D"])
PTRM_OWN

### Under the harness

The gate re-opens the puzzles with `q < 0` and the fallback returns the grid of highest `q`. Without the stopping
rule, the `K'` rollouts are all run on every re-opened puzzle; with it, they are read in order until one has
`q >= 0`. Cost in blocks per puzzle, one block per `T` steps.

In [ ]:
def harness(ok, q, k, D):
    """PTRM under the gate and the fallback, without and with the stopping rule: correctness per puzzle and cost."""
    ok, q = ok[:k], q[:k]
    n = ok.shape[1]
    cols, gate = np.arange(n), q0 < Q_THR
    cand_ok, cand_q = np.concatenate([ok0[None, :], ok]), np.concatenate([q0[None, :], q])
    gated = np.where(gate, cand_ok[cand_q.argmax(0), cols], ok0)
    hit = q >= Q_THR
    last = np.where(hit.any(0), hit.argmax(0), k - 1)
    read = np.concatenate([np.ones((1, n), bool), np.arange(k)[:, None] <= last[None, :]])
    stopped = np.where(gate, cand_ok[np.where(read, cand_q, -np.inf).argmax(0), cols], ok0)
    return gated, 1.0 + gate.mean() * (k * D / T), stopped, 1.0 + float(((last + 1) * gate).sum()) * D / T / n


rows = {}
for D in PTRM_D:
    for k in K_LIST:
        gated, gated_cost, stopped, stopped_cost = zip(*(harness(ok, q, k, D) for ok, q in ROLLOUTS[D]))
        (a, sd, _), (sa, ssd, _) = accuracy(gated), accuracy(stopped)
        rows[(k, D)] = {"cost": gated_cost[0], "accuracy": a, "sd": sd, "stopping rule cost": np.mean(stopped_cost),
                        "stopping rule accuracy": sa, "stopping rule sd": ssd}
PTRM_HARNESS = pd.DataFrame.from_dict(rows, orient="index").rename_axis(["K'", "D"])
PTRM_HARNESS